# Ejercicio 7: La reconstrucción como proyección ortogonal

En este ejercicio interpretamos geométricamente la reconstrucción por mínimos cuadrados obtenida para las mediciones con ruido correspondiente a $\sigma=0.06$. Calculamos el residuo y verificamos que es ortogonal al espacio columna de $A_7$. Finalmente, diferenciamos la proyección tomográfica de la proyección ortogonal utilizada en mínimos cuadrados.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

##### Código de ejercicios anteriores

In [2]:
# Definimos el fantoma
X_fantoma = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
    [0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

# Vectorizamos el fantoma por filas
x_fantoma = X_fantoma.reshape(-1)


def mascara_rayo(n, direccion, s):
    # Creamos una matriz de n por n llena de ceros
    matriz = np.array([[0] * n for _ in range(n)])

    # Recorremos todos los píxeles
    for i in range(n):
        for j in range(n):
            # Verificamos si el píxel pertenece al rayo
            if direccion[0] * i + direccion[1] * j == s:
                matriz[i, j] = 1

    return matriz


def matriz_proyeccion(n, direcciones):
    # Guardamos las máscaras vectorizadas
    matriz = []

    # Guardamos la dirección y el desplazamiento de cada rayo
    rayo = []

    for direccion in direcciones:
        valores_s = []

        # Calculamos todos los valores posibles de s
        for i in range(n):
            for j in range(n):
                valores_s.append(direccion[0] * i + direccion[1] * j)

        # Eliminamos valores repetidos y los ordenamos
        valores_s = sorted(set(valores_s))

        # Construimos una máscara para cada valor de s
        for s in valores_s:
            rayo.append((tuple(direccion), s))
            mascara = mascara_rayo(n, direccion, s)
            matriz.append(mascara.reshape(-1))

    return np.array(matriz), rayo


# Definimos las siete direcciones utilizadas por A7
D7 = [(1, 0), (0, 1), (1, 1), (1, -1), (2, 1), (1, 2), (3, 1)]

# Construimos la matriz de proyección A7
A7, rayos_A7 = matriz_proyeccion(8, D7)

# Calculamos las mediciones ideales
b7 = A7 @ x_fantoma

# Fijamos la semilla para reproducir el mismo ruido del ejercicio anterior
np.random.seed(18)

# Generamos el vector aleatorio
z = np.random.normal(0, 1, len(b7))

# Calculamos el tamaño típico de las mediciones
escala = np.sqrt(np.mean(b7**2))

# Construimos el ruido correspondiente a sigma = 0.06
sigma = 0.06
n_sigma = sigma * escala * z

# Sumamos el ruido a las mediciones ideales
b_sigma = b7 + n_sigma

# Reconstruimos la imagen mediante mínimos cuadrados
x_raya_sigma = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]

## a) Verificación de la condición de ortogonalidad

Calculamos el residuo $r_\sigma=b_\sigma-A_7\widehat{x}_\sigma$, que representa la diferencia entre las mediciones ruidosas y las mediciones producidas por la imagen reconstruida. Luego calculamos $\|A_7^Tr_\sigma\|$ para verificar si el residuo es ortogonal a las columnas de $A_7$.

In [ ]:
# Calculamos el residuo de la reconstrucción
residuo = b_sigma - A7 @ x_raya_sigma

# Calculamos A7 transpuesta por el residuo
producto = A7.T @ residuo

# Calculamos la norma para verificar si el resultado es aproximadamente cero
norma = np.linalg.norm(producto)

print("Norma de A7.T @ residuo:", norma)

Norma de A7.T @ residuo: 3.679816846255149e-14


Obtenemos $\|A_7^Tr_\sigma\|=3.68\times10^{-14}\approx0$. Este valor es extremadamente pequeño y puede considerarse numéricamente igual a cero. Por lo tanto, verificamos que el residuo es aproximadamente ortogonal a todas las columnas de $A_7$.

## b) Interpretación del residuo

**¿Por qué el resultado anterior indica que el residuo es ortogonal a todas las columnas de $A_7$ y que $A_7\widehat{x}_\sigma$ es la proyección ortogonal de $b_\sigma$ sobre $\operatorname{Col}(A_7)$?**

Las columnas de $A_7$ son las que generan todo el $Col(A_7)$. Hacer la multiplicación de $A^T_7 {\times} r_{\sigma}$ es hacer el producto escalar de $r_{\sigma}$ contra las columnas de $A_7$. Sabemos que si el producto interno de dos vectores es $0$, son ortogonales. Como da $0$ con todas las columnas, el residuo $r_{\sigma}$ forma un ángulo de 90° con todo el subespacio $Col(A_7)$. 

Por definicion, el único punto de un subespacio cuya distancia a un vector externo forma una recta perpendicular es la proyeccion ortogonal. Por lo tanto, $A_7{\hat x}_{\sigma}$ es la proyección ortogonal de $b_{\sigma}$ sobre $Col(A_7)$

## c) Comparación entre los dos tipos de proyección

**¿Cuál es la diferencia entre las proyecciones tomográficas medidas por el escáner y la proyección ortogonal que aparece en mínimos cuadrados?**

La proyección tomográfica (del escáner) es el procedimiento en el que una serie de rayos X paralelos atraviesa al paciente en un ángulo determinado y proyecta la atenuación del cuerpo sobre una línea de detectores. Transforma un corte en 2D (como se menciona en el modelo) a señales 1D.

La proyección ortogonal que aparece en mínimos cuadrados es una operación matemática que toma el vector con ruido ${b_{\sigma}}$ que no coincide con ninguna imagen y lo "baja perpendicularmente" sobre el subespacio de mediciones válidas para encontrar el vector ${A_7\hat x_\sigma}$ mas cercano posible.